In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import sys
import os
import torch
from peft import get_peft_model, LoraConfig, TaskType
from peft.tuners.lora import LoraModel

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [15]:
from src.trainer import LoRASmoothTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets, create_holdout_set
from src.utils.general import InContextHead, set_seed
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

### Task Incremental Learning

In [10]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_fully_connected_model(head=head, dense_layers=3, seed=SEED, device='cuda', input_dim=28*28)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [29]:
trainer = SimpleTrainer(
    model, paradigm="TIL", seed=SEED
)
trainer.train(train_tasks[0], val_tasks[0], context_id=0)
trainer.test(test_tasks, context_list=list(range(5)))

Training Epochs:   0%| | 0/5 [00:00<?, ?it/s, train_loss=0.6861, val_loss=0, val

Training Epochs: 100%|█| 5/5 [00:01<00:00,  3.04it/s, train_loss=0.3259, val_los


Test Results: [(0.0935, 0.9705), (1.6868, 0.357), (4.0678, 0.0), (3.3294, 0.5132), (1.0024, 0.476)] (Avg: (2.0360, 0.4633))


[(0.0935, 0.9705),
 (1.6868, 0.357),
 (4.0678, 0.0),
 (3.3294, 0.5132),
 (1.0024, 0.476)]

In [30]:
train_tasks[0], buffer_task = create_holdout_set(train_tasks[0], holdout_size=2500)

In [31]:
lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=["1", "3", "5", "7"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

peft_model = LoraModel(trainer.model, lora_config, adapter_name='default')

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(peft_model.parameters(), lr=1e-3)

loader = torch.utils.data.DataLoader(buffer_task, batch_size=256, shuffle=True)

peft_model.train()
total_loss = 0
for epoch in range(5):
    for x,y in loader:
        x, y = x.to('cuda'), y.to('cuda')
        optimizer.zero_grad()
        out = peft_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Total loss for epoch: {total_loss:.4f}")

Total loss for epoch: 6.9163
Total loss for epoch: 13.8288
Total loss for epoch: 20.7338
Total loss for epoch: 27.6265
Total loss for epoch: 34.5024


In [32]:
smooth_trainer = LoRASmoothTrainer(
    trainer.model,
    peft_model,
    smooth_cheat = 1000,
    smooth_delta=0.1,
    paradigm="TIL",
    seed=SEED
)

smooth_trainer.compute_lid(test_tasks[0], context_id=0)

100%|████████████████████████████████████████| 100/100 [00:00<00:00, 560.57it/s]


Inflate: 1.0, Estimated success: 0.0000, Certified radius: -inf


UnboundLocalError: local variable 'max_inflate' referenced before assignment